# CourtListener search → DataFrame (HCDE 530)

Uses the **Week 4** folder and reads **`COURTLISTENER_API_TOKEN`** from **`week 4/.env`** (same variable as `week4_public_api.py`). Supports `KEY=value` or `export KEY=value` in `.env`.

Install packages into the **same** interpreter as this notebook (global kernel): `python3 -m pip install -r "week 4/requirements.txt"` — no virtualenv required.

Calls CourtListener **Legal Search API v4**: `GET /api/rest/v4/search/` with `type=o` (opinion clusters).

The API returns **`caseName`**, **`judge`**, and **`dateFiled`**. Plaintiff and defendant are **not** separate API fields; we **parse** them from `caseName` when it looks like *Party A v. Party B* (otherwise those columns are missing).


In [1]:
from __future__ import annotations

import json
import os
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import pandas as pd

# Find folder that contains `week 4/.env` or `.env` (kernel cwd varies)
HERE = Path.cwd()
for candidate in (HERE, HERE / "week 4", HERE.parent / "week 4"):
    if (candidate / ".env").is_file():
        HERE = candidate
        break
ENV_PATH = HERE / ".env"
TOKEN_ENV = "COURTLISTENER_API_TOKEN"
SEARCH_URL = "https://www.courtlistener.com/api/rest/v4/search/"


def load_dotenv_file(path: Path) -> None:
    if not path.is_file():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key = key.strip()
        if key.lower().startswith("export "):
            key = key[7:].strip()
        val = val.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = val


def split_parties(case_name: str) -> tuple[str, str]:
    if not case_name or not isinstance(case_name, str):
        return "", ""
    for sep in (" v. ", " V. ", " vs. ", " VS. ", " v ", " V "):
        if sep in case_name:
            left, right = case_name.split(sep, 1)
            return left.strip(), right.strip()
    return "", ""


def courtlistener_search(query: str, *, opinion_type: str = "o", token: str | None) -> dict:
    params = {"q": query, "type": opinion_type}
    url = SEARCH_URL + "?" + urllib.parse.urlencode(params)
    headers = {"Accept": "application/json; indent=2"}
    if token:
        headers["Authorization"] = f"Token {token}"
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        detail = e.read().decode("utf-8", errors="replace")[:800]
        raise RuntimeError(f"CourtListener HTTP {e.code}: {detail}") from e


load_dotenv_file(ENV_PATH)
token = os.environ.get(TOKEN_ENV)
if not token:
    raise RuntimeError(
        f"Missing {TOKEN_ENV}. Add to {ENV_PATH} (see .env.example). Do not commit .env."
    )

QUERY = "Miranda v. Arizona"
raw = courtlistener_search(QUERY, token=token)
results = raw.get("results") or []
print("total matches:", raw.get("count"))
print("rows on first page:", len(results))


RuntimeError: Missing COURTLISTENER_API_TOKEN. Add to /Users/rnr6/Documents/HCDE/hcde530/week 4/.env (see .env.example). Do not commit .env.

In [ ]:
rows: list[dict[str, object]] = []
for item in results[:25]:
    case = (item.get("caseName") or item.get("caseNameFull") or "").strip()
    plaintiff, defendant = split_parties(case)
    judge = (item.get("judge") or "").strip()
    panel = item.get("panel_names") or []
    if not judge and panel:
        judge = "; ".join(str(p) for p in panel if p)
    date_dec = item.get("dateFiled") or item.get("dateArgued")

    rows.append(
        {
            "case": case,
            "plaintiff": plaintiff or pd.NA,
            "defendant": defendant or pd.NA,
            "judge": judge or pd.NA,
            "date_of_decision": date_dec or pd.NA,
        }
    )

df = pd.DataFrame(rows)
df


### Notes

- **Pagination**: follow the `next` URL (cursor) for more pages; cache is ~10 minutes per CourtListener docs.
- **Token**: same `COURTLISTENER_API_TOKEN` as `week4_public_api.py` in `week 4/.env`.
- **Docs**: [Legal Search API](https://www.courtlistener.com/help/api/rest/search/)
